In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [10]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

In [11]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [12]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [13]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [14]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [15]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Football'}, config=config1)

{'topic': 'Football',
 'joke': 'Why did the football team go to the bank?\n\nBecause they wanted to get their **quarter‑back**!',
 'explanation': '**The joke:**  \n*Why did the football team go to the bank?*  \n*Because they wanted to get their **quarter‑back**!*\n\n---\n\n### 1. What a “quarter‑back” is in football  \n- In American football, the **quarterback (QB)** is the player who lines up behind the center and usually receives the ball on the first “snap.”  \n- The quarterback is the team’s primary passer and often the leader of the offense.\n\n### 2. What a “quarter” is at a bank  \n- A **quarter** is a U.S. coin worth 25\u202fcents.  \n- When you go to a bank you can withdraw cash, including quarters.\n\n### 3. The wordplay (the “pun”)  \n- The phrase **“quarter‑back”** can be split into two ordinary English words: **quarter** + **back**.  \n- If you interpret it literally, a “quarter‑back” could be someone who goes **back** to get a **quarter** (the coin).  \n- So the joke pret

In [19]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football team go to the bank?\n\nBecause they wanted to get their **quarter‑back**!', 'explanation': '**The joke:**  \n*Why did the football team go to the bank?*  \n*Because they wanted to get their **quarter‑back**!*\n\n---\n\n### 1. What a “quarter‑back” is in football  \n- In American football, the **quarterback (QB)** is the player who lines up behind the center and usually receives the ball on the first “snap.”  \n- The quarterback is the team’s primary passer and often the leader of the offense.\n\n### 2. What a “quarter” is at a bank  \n- A **quarter** is a U.S. coin worth 25\u202fcents.  \n- When you go to a bank you can withdraw cash, including quarters.\n\n### 3. The wordplay (the “pun”)  \n- The phrase **“quarter‑back”** can be split into two ordinary English words: **quarter** + **back**.  \n- If you interpret it literally, a “quarter‑back” could be someone who goes **back** to get a **quarter** (the coin).  \

In [24]:
# to get all state(intermediate + final state)
list(workflow.get_state_history(config1))

# https://youtu.be/YIv_GDxJqbA?si=NWtRJn7al9scYadB&t=6859

[StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football team go to the bank?\n\nBecause they wanted to get their **quarter‑back**!', 'explanation': '**The joke:**  \n*Why did the football team go to the bank?*  \n*Because they wanted to get their **quarter‑back**!*\n\n---\n\n### 1. What a “quarter‑back” is in football  \n- In American football, the **quarterback (QB)** is the player who lines up behind the center and usually receives the ball on the first “snap.”  \n- The quarterback is the team’s primary passer and often the leader of the offense.\n\n### 2. What a “quarter” is at a bank  \n- A **quarter** is a U.S. coin worth 25\u202fcents.  \n- When you go to a bank you can withdraw cash, including quarters.\n\n### 3. The wordplay (the “pun”)  \n- The phrase **“quarter‑back”** can be split into two ordinary English words: **quarter** + **back**.  \n- If you interpret it literally, a “quarter‑back” could be someone who goes **back** to get a **quarter** (the coin).  

In [25]:

config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Cricket'}, config=config2)

{'topic': 'Cricket',
 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the scores were *up* and they wanted to *reach* a higher *run* rate! 🏏😄',
 'explanation': '**What the joke is playing with**\n\n| Cricket term | Everyday meaning | How the joke twists it |\n|--------------|------------------|------------------------|\n| **Score** (or “the scores”) | The number of runs a team has made. | The joke pretends “the scores were *up*” means the numbers are high **and** that something is literally high in the air. |\n| **Run rate** | The average number of runs a team scores per over (runs\u202f÷\u202fovers). A higher run rate is better. | “Reach a higher run rate” sounds like you need to *physically* reach something that’s higher up, so a ladder would help. |\n| **Up** | “Up” can mean “increased” (e.g., “the scores are up”) **or** “above” in space. | The double meaning lets the joke treat “up” as a direction that you can climb. |\n| **Ladder** | A tool for

In [22]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the batsmen were *always* trying to reach new heights! 🏏😆', 'explanation': '**What makes the joke funny?**  \n\n1. **The literal image** – A ladder is an object you use to climb up and reach something that’s higher than you can reach with your own height or arms. If you picture a cricket team actually hauling a ladder onto the field, the mental picture is absurd and a little silly, which already sets up a comedic tone.\n\n2. **The cricket‑specific meaning of “batsmen”** – In cricket, a batsman’s job is to hit the ball with the bat. Good batsmen try to *score* a lot of runs, and one way they do that is by hitting the ball high into the air (a “lofted” shot) or by “reaching new heights” in terms of performance (higher scores, better technique, more impressive feats).\n\n3. **The wordplay on “reach new heights”** –  \n   * *Reach new heights* is a common idiom mea

In [23]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the batsmen were *always* trying to reach new heights! 🏏😆', 'explanation': '**What makes the joke funny?**  \n\n1. **The literal image** – A ladder is an object you use to climb up and reach something that’s higher than you can reach with your own height or arms. If you picture a cricket team actually hauling a ladder onto the field, the mental picture is absurd and a little silly, which already sets up a comedic tone.\n\n2. **The cricket‑specific meaning of “batsmen”** – In cricket, a batsman’s job is to hit the ball with the bat. Good batsmen try to *score* a lot of runs, and one way they do that is by hitting the ball high into the air (a “lofted” shot) or by “reaching new heights” in terms of performance (higher scores, better technique, more impressive feats).\n\n3. **The wordplay on “reach new heights”** –  \n   * *Reach new heights* is a common idiom me